In [2]:
from ultralytics import YOLO
import cv2
import os
import numpy as np
import tensorflow as tf
from sort import Sort
import time 
from collections import deque

c:\Python312\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


In [3]:
# Cargar el modelo preentrenado YOLOv8
modelo = YOLO("yolov8n.pt")
tracker = Sort() # Crear un tracker de objetos
modeloSus = tf.keras.models.load_model('../AlertGuard/modelSuspicious.h5')
#classes = ['no_violencia', 'violencia']
print("Modelo cargado correctamente")

Modelo cargado correctamente


In [4]:
# Ruta de la carpeta con videos
carpeta_videos = "../videos/normal"
frames_personas = "../personasNormal" # Carpeta para guardar los frames con personas
# Obtener la lista de videos en la carpeta
videos = [os.path.abspath(os.path.join(carpeta_videos, f)) for f in os.listdir(carpeta_videos) 
          if f.endswith(('.mp4', '.avi', '.mov'))]
print("Videos encontrados:", videos)

# Crear la carpeta para guardar los frames con personas
if not os.path.exists(frames_personas):
    os.makedirs(frames_personas)

Videos encontrados: ['c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\10_1_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\10_2_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\10_3_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\11_1_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\11_2_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\11_3_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\12_1_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\12_2_crop.mp4', 'c:\\Users\\valen\\OneDrive\\Documentos\\UNIVERSIDAD\\proyecto\\modeloVideo\\videos\\normal\\12_3_crop.mp4'

In [7]:
SEQUENCE_LENGTH = 100 
IMG_SIZE = 112

# Diccionario para almacenar secuencias de imágenes de cada persona detectada
sequences = {}

def preprocesar_imagen(persona):
    """Convierte la imagen al formato esperado por el modelo."""
    persona = cv2.resize(persona, (IMG_SIZE, IMG_SIZE))  # Redimensionar
    persona = persona.astype(np.float32) / 255.0  # Normalizar
    return persona

def mostrar_video_con_yolo(video_path):
    cap = cv2.VideoCapture(video_path)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break  # Salir si no hay más frames

        # Aplicar YOLO para detección de personas
        results = modelo(frame)
        for r in results:
            personas = np.where(r.boxes.conf.cpu().numpy() > 0.5)[0]
            boxes = r.boxes.xyxy.cpu().numpy()[personas].astype(int)
            tracks = tracker.update(boxes).astype(int)

            for xmin, ymin, xmax, ymax, track_id in tracks:
                cv2.putText(frame, f"Id: {track_id}", (xmin, ymin - 10), cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 2)
                cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)

                # Extraer y preprocesar la imagen de la persona detectada
                if ymax > ymin and xmax > xmin:  # Evitar imágenes vacías
                    persona = frame[ymin:ymax, xmin:xmax]
                    persona = preprocesar_imagen(persona)

                    # Agregar la imagen a la secuencia del track_id
                    if track_id not in sequences:
                        sequences[track_id] = deque(maxlen=SEQUENCE_LENGTH)  # Lista FIFO

                    sequences[track_id].append(persona)

                    # Si ya tenemos suficientes frames en la secuencia, hacemos la predicción
                    if len(sequences[track_id]) == SEQUENCE_LENGTH:
                        secuencia_np = np.array(sequences[track_id])  # Convertir a numpy array
                        secuencia_np = np.expand_dims(secuencia_np, axis=0)  # Agregar dimensión de batch

                        # Hacer la predicción
                        prediccion = modeloSus.predict(secuencia_np)[0]
                        clasePrediccion = np.argmax(prediccion)
                        cv2.putText(frame, f"Prediccion: {clasePrediccion}", (xmin, ymin - 10), cv2.FONT_HERSHEY_PLAIN, 2, (0, 255, 0), 2)

        # Mostrar el frame con detecciones
        cv2.imshow("YOLO - Detección de Personas", frame)

        # Salir con la tecla 'q'
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

# Procesar un video
mostrar_video_con_yolo(videos[2])


0: 480x640 (no detections), 66.5ms
Speed: 2.3ms preprocess, 66.5ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 2 persons, 71.7ms
Speed: 3.0ms preprocess, 71.7ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 67.0ms
Speed: 0.0ms preprocess, 67.0ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 86.3ms
Speed: 0.0ms preprocess, 86.3ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 79.4ms
Speed: 0.0ms preprocess, 79.4ms inference, 0.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 1 person, 71.4ms
Speed: 0.0ms preprocess, 71.4ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 67.8ms
Speed: 2.0ms preprocess, 67.8ms inference, 1.0ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 (no detections), 65.3ms
Speed: 2.0ms preprocess, 65.3ms inference, 1.0ms postprocess 